# Exploration of DM+D codes structure in prescriptions data

In [ ]:
import pyspark
import dxpy
import hail as hl

In [ ]:
sc = pyspark.SparkContext()
spark = pyspark.sql.SparkSession(sc)

In [ ]:
hl.init(sc=sc, default_reference='GRCh38')

#### Envinroment setup

In [ ]:
from datetime import datetime
print(f'Timestamp: {datetime.now()}')
print(f'Instance type: {dxpy.describe(dxpy.JOB_ID)["instanceType"]}')
print(f'Hail version: {hl.version()}')
print(f'Spark version: {spark.version}')

### Input database configuration and loading

In [ ]:
db_name = 'clinical_phenos'
full_tb_name = 'full_phenos_hail_0.2.116.ht' #'full_phenos_details_smp_0100.ht'

In [ ]:
db_uri = dxpy.find_one_data_object(name=f"{db_name}", classname="database", project=dxpy.PROJECT_CONTEXT_ID)['id']
url = f"dnax://{db_uri}/{full_tb_name}"
full = hl.read_table(url)

### Checking dataset size

In [ ]:
full.count()

In [ ]:
%time system_df = full.filter(full.system == 'dmd').cache()
system_df.count()

### Adding neccessary helpers

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
from datetime import datetime

def perform_count_aggregation(grouped_by, input_data, aggregated = False):
    if aggregated:
        aggregated = grouped_by
    else:
        aggregated = grouped_by.aggregate(occurences=hl.agg.count())
    data_count = input_data.count()
    aggregated = aggregated.annotate(share = hl.format('%.3f%%', aggregated.occurences / hl.float(data_count) * 100))
    aggregated = aggregated.order_by(-aggregated.occurences).cache()
    return aggregated
    
def show_aggregated_examples(aggredated, source_data, column, sample_sz):
    source_data_cols = list(source_data.row.keys())
    source_data_cols.remove(column)
    source_data_cols.insert(0, column)
    aggregated_py = aggregated.collect()
    rand_seed = int(datetime.now().timestamp())
    source_data = source_data.annotate(rand = hl.rand_unif(0, 1, seed = rand_seed)).order_by('rand').cache()
    joined = source_data.head(0)
    for row in aggregated_py:
        joined = joined.union(source_data.filter(source_data[column] == row[column]).head(sample_sz))
    joined = joined.cache()
    joined = joined.key_by(column).join(aggredated.key_by(column), how = 'left')
    joined = joined.order_by(-joined.occurences).select(*source_data_cols).cache()
    joined.show(-1)

def aggregated_bar_plot(aggregated, column):
    aggregated_pd = aggregated.to_pandas()
    aggregated_pd['share_pct'] = aggregated_pd['share'].str.rstrip('%').astype(float)
    
    plt.figure(figsize=(8, 4))
    plt.bar(aggregated_pd[column], aggregated_pd['share_pct'], color = plt.cm.viridis(np.linspace(0, 1, 5)))
    plt.ylabel('Share %')

## Checking DM+D codes length distribution

In [ ]:
%time system_df = system_df.annotate(code_len = hl.len(system_df.code)).cache()

In [ ]:
%time aggregated = perform_count_aggregation(system_df.group_by('code_len'), system_df)
aggregated.show(-1)

In [ ]:
# show_aggregated_examples(aggregated, system_df, 'code_len', 5)

In [ ]:
aggregated_bar_plot(aggregated.annotate(code_len = hl.str(aggregated.code_len)), 'code_len')
plt.title('DM+D prescriptions code length')
plt.show()

## Checking Read v2 codes formats

**Notice**: Following findings about DM+D codes structure are mainly data-driven (Biobank clinical and prescriptions data and Read lookups/dictionaries).

DM+D codes format in Biobank data falls into a few categories:


| Class name           | Format as regular expression                     | Example   | Description                                                                 |
|----------------------|--------------------------------------------------|-----------|-----------------------------------------------------------------------------|
| `dmd_lookup`                 | `^\d{13,17}$`                        | `2913211000001105`   | 13-17 digit codes which mostly can be found in Biobank DM+D lookup. |
| `snomed`       | `^\d{9}$`          | `323510009` | 9 digit SNOMED CT code. |
| `unknown_18_digit`       | `^\d{18}$`                      | `182985001000027101`   | Unknown 18 digit code (not found in anywhere). |
| `unknown_68461003`        | `^68461003$`                       | `68461003`   | Mysterious `68461003` code.          |
| `missing`      | `^0$`                         | `0`   | Missing code placeholder. |                      |


**Important notice**: The codes used to identify DM+D concepts are of the same form as those used in SNOMED CT and thus conform to the same specification.

SNOMED CT codes online lookup: https://dmd-browser.nhsbsa.nhs.uk/code-lookup.

In [ ]:
code_formats = {
    'dmd_lookup': (r'^\d{13,17}$', '2913211000001105'),
    'snomed': (r'^\d{9}$', '323510009'),
    'unknown_18_digit': (r'^\d{18}$', '182985001000027101'),
    'unknown_68461003': (r'^68461003$', '68461003'),
    'missing': (r'^0$', '0'),
}

In [ ]:
%%time
hl_code_formats = hl.literal([(code_formats[k][0], k) for k in code_formats.keys()])
system_df = system_df.annotate(
    code_format = hl.or_else(hl.find(lambda cformat: system_df
.code.matches(cformat[0]), hl_code_formats), hl.literal(('.+', 'other')))[1]
).cache()

#### Examining unknown DM+D code format records

In [ ]:
other = system_df.filter(system_df.code_format == 'other').cache()
other.count()

In [ ]:
other.show()

#### DM+D code formats prescriptions share

In [ ]:
%time aggregated = perform_count_aggregation(system_df.group_by('code_format'), system_df)
aggregated.show(-1)

In [ ]:
aggregated_bar_plot(aggregated, 'code_format')
plt.title('DM+D prescriptions code formats')
plt.xticks(rotation=60)
plt.show()

#### Prescriptions examples of particular DM+D code formats

In [ ]:
%time show_aggregated_examples(aggregated, system_df, 'code_format', 5)